# Part A6 — The Large Chess Network (Free Internet Chess Server)

This notebook finds the **top-10 most central players** in the Free Internet
Chess Server (FICS) game network and **draws a meaningful slice** of it.

The whole point of this task is that the network is *enormous*: about
**429,747,477 edges** (one row for each direction of each game ever played).
That is far too big to load into an ordinary in-memory graph library. So a big
part of the work is *how you tame the size*. I explain that carefully below.

I write the way I like explanations: **long but plain**. Every technical term is
defined in everyday words the first time it shows up, and I build each idea from
the ground up instead of just stating a conclusion.

**Data source:** Free Internet Chess Server network —
<http://dynamics.cs.washington.edu/nobackup/chess/fcis.tar.gz>
(a 6.85 GB gzip-compressed tar archive; "gzip" is a common file-compression
format, and a "tar archive" is a single file that bundles many files together,
like a zip).

**Libraries used in this notebook:**
- `polars` — a very fast table/dataframe library (think of a spreadsheet engine
  in code) that can read data in a *streaming* way, meaning it processes the file
  in small chunks instead of loading it all into memory at once.
- `python-igraph` (imported as `igraph`) — a graph library written in C, so it is
  fast on big graphs. We use it for PageRank, degree, and strength.
- `networkx` — a pure-Python graph library; here we only use it to lay out and
  draw the small picture (it is slower but its drawing tools are convenient).
- `numpy` / `pandas` — number crunching and small result tables.
- `scipy` — one statistics function (Spearman rank correlation).
- `matplotlib` — the plotting library. The cluster has **no screen**, so we force
  the **"Agg" backend** (a backend is just the engine matplotlib uses to render;
  "Agg" draws straight to an image file in memory, no window needed).


In [1]:
# --- Setup: imports, headless plotting, fixed random seed ---
import sys, time
sys.path.insert(0, "/home/mickaelz/Network analysis/src")
import na_utils as na           # shared helpers: na.set_style(), na.save_fig()

import numpy as np
import polars as pl             # fast streaming dataframes
import igraph                   # fast C graph library (PageRank etc.)
import networkx as nx           # used only for drawing the small slice
import pandas as pd
from scipy.stats import spearmanr

np.random.seed(42)              # fix the seed so results are reproducible
na.set_style()                  # forces matplotlib's headless 'Agg' backend
import matplotlib.pyplot as plt

BASE  = "/home/mickaelz/Network analysis"
STATS = f"{BASE}/data/chess/player_stats.parquet"   # FULL-graph per-player games
EDGES = f"{BASE}/data/chess/edges_top.parquet"      # top-5000 induced subgraph
print("igraph", igraph.__version__, "| networkx", nx.__version__,
      "| polars", pl.__version__)


igraph 1.0.0 | networkx 3.2.1 | polars 1.36.1


## 1. How I handled the network size (the core deliverable)

### The problem, in concrete numbers

The raw download expands into a folder `data/chess/FCIS/` containing two plain
text files in **CSV** format (CSV = "comma-separated values", a text table where
each line is a row and commas separate the columns):

- `fcis_chess.interactions.csv` — **15.4 gigabytes**, with **429,747,477 rows**.
  Each row is `datetime, src_id, dst_id`: a timestamp, the player who is the
  *source* of that interaction, and the player who is the *destination*. This is
  an **edge list** — literally a long list, one line per edge, naming the two
  endpoints. Crucially, **each game is stored twice**: once as `(a, b)` and once
  as `(b, a)`. So 429.7 million rows correspond to about **214.9 million actual
  games**.
- `fcis_chess.vertices.csv` — one row per player: `mindate, v_id, maxdate`
  (first activity date, the player's username, last activity date). There are
  **519,584 players**. Players are identified by their **username**, a short
  text handle like `mscp` or `GriffyJr` — not a number.

A quick reality check on why this cannot go into memory the naive way. NetworkX
(a pure-Python graph library) stores each edge as Python objects inside nested
dictionaries. A rough rule of thumb is that this costs on the order of a few
hundred bytes per edge. Multiply that by 429 million and you are deep into the
**hundreds of gigabytes** of RAM — vastly more than the machine has. So
"`G = nx.read_edgelist(...)`" is simply off the table.

### The idea: *streaming aggregation* instead of loading the graph

Here is the key trick. For the questions we actually need to answer — "who plays
the most games?" and "what does the busy core of the network look like?" — we do
**not** need every one of the 429 million individual edges sitting in memory at
once. We only need *summaries*. And a summary can be computed by reading the file
**once, in small chunks**, keeping only a tiny running tally, and throwing each
chunk away after we have counted it.

That technique is called **streaming aggregation**. "Streaming" means we read the
file like water flowing through a pipe — a little at a time — rather than pouring
the whole lake into a bucket. "Aggregation" means we *combine many rows into a
few summary numbers* (counts, sums) as they flow past.

> **Tiny analogy.** Imagine you must count how many cars of each colour drive
> past your window in a day. You do *not* need to photograph all 100,000 cars and
> store the photos. You keep a small notepad with one tally mark per colour and
> update it as each car passes. At the end you have the counts, and you never
> needed a warehouse of photos. Streaming aggregation is exactly this: the file
> is the stream of cars, the running counts are the notepad.

`polars` does this for us with `scan_csv(...).collect(engine="streaming")`.
`scan_csv` does **not** read the file immediately — it builds a *plan* (a recipe)
of what we want. `collect(engine="streaming")` then runs that plan in a
memory-bounded, chunk-by-chunk way.

### This heavy step was already run as a batch job (and is NOT re-run here)

Reading 15.4 GB still takes a couple of minutes, so the aggregation was run
**once**, ahead of time, as a **SLURM batch job**. (SLURM is the scheduling
system on the compute cluster: you hand it a script, it finds a free machine,
runs your job there, and saves the output. A "batch job" just means "run this
unattended and tell me when it's done.") The job asked for 8 CPU cores and 48 GB
of RAM, finished in **1 minute 51 seconds**, and peaked at only about **22 GB**
of memory — comfortably under the limit, precisely *because* we stream instead of
loading everything. The exact code is in `scripts/chess_aggregate.py` and the job
description in `scripts/chess_aggregate.sbatch`.

That job produced **two small files** that this notebook loads instantly. They
are stored in **Parquet** format (Parquet = a compact, column-oriented binary
file format for tables; "column-oriented" means it stores all values of one
column together, which makes reading and compressing very efficient):

1. `data/chess/player_stats.parquet` — columns `(player, games)`. The **exact**
   total number of games each of the **519,583** players ever played, computed
   over the **full** 429-million-edge graph. Nothing sampled, nothing
   approximated.
2. `data/chess/edges_top.parquet` — columns `(src_id, dst_id, w)`. The **induced
   subgraph** on the **top-5000 most active players**: every game played
   *between two of those 5000 players*, with `w` = how many games that ordered
   pair played. About **7.99 million** directed edges.

I unpack what "weighted degree" and "induced subgraph" mean just below, right
where we use them.

### Why this two-file design is the right call (justification)

- **The per-player game counts are EXACT, not sampled.** This is the cleanest
  trick. Because every game appears as both `(a, b)` and `(b, a)`, if we simply
  group the 429M rows by the *source* column and count rows per source, each
  player's count is exactly their total number of games. Grouping by a single
  column produces only ~519k groups — a tiny tally that fits easily in memory.
  So our *primary* centrality measure is computed on the **whole** graph with no
  approximation at all.

- **The top-5000 induced subgraph captures the players who matter for
  structure.** "Induced subgraph" means: pick a set of players, then keep *only*
  the edges whose **both** endpoints are inside that set (you "induce" the
  sub-network from the chosen nodes). We pick the 5000 busiest players. Why is
  that a sound choice rather than an arbitrary sample? Because in interaction
  networks the most structurally central players are *overwhelmingly* among the
  most active — you cannot be a hub that everything flows through if you barely
  play. So restricting structural analysis (PageRank, degree) to the busy core
  keeps the players who would top any centrality ranking, while shrinking 429M
  edges down to a graph small enough to run real algorithms on in under a second.
  We will literally *measure* this claim later (the PageRank top-10 turns out to
  share 9 of its 10 names with the exact games top-10).

- **We never re-stream the 15 GB CSV inside this notebook.** The cell below shows
  the streaming code for the record, but it is guarded by `RUN_HEAVY = False` so
  it does **not** execute. The notebook runs entirely from the two small Parquet
  files, so it is fast and reproducible.


In [2]:
# The heavy streaming aggregation, shown for the record but NOT executed.
# It is exactly scripts/chess_aggregate.py. Flip RUN_HEAVY to True only on a
# machine that has the 15 GB CSV and lots of RAM/time.
RUN_HEAVY = False

if RUN_HEAVY:
    SRC = f"{BASE}/data/chess/FCIS/fcis_chess.interactions.csv"
    schema = {"datetime": pl.Utf8, "src_id": pl.Utf8, "dst_id": pl.Utf8}

    # Step 1 — EXACT games per player. Group by a SINGLE column (src_id): only
    # ~519k groups, so the running tally is tiny. Streaming keeps memory bounded.
    stats_full = (pl.scan_csv(SRC, schema_overrides=schema)
                    .group_by("src_id").agg(pl.len().alias("games"))
                    .rename({"src_id": "player"})
                    .sort("games", descending=True)
                    .collect(engine="streaming"))
    stats_full.write_parquet(STATS)

    # Step 2 — induced subgraph on the top-5000 busiest players. Stream again,
    # keep only rows whose BOTH endpoints are in the top set, then count pairs.
    TOP_K = 5000
    top_set = pl.Series(stats_full.head(TOP_K)["player"].to_list())
    edges_full = (pl.scan_csv(SRC, schema_overrides=schema)
                    .filter(pl.col("src_id").is_in(top_set)
                            & pl.col("dst_id").is_in(top_set))
                    .group_by(["src_id", "dst_id"]).agg(pl.len().alias("w"))
                    .collect(engine="streaming"))
    edges_full.write_parquet(EDGES)

print("RUN_HEAVY =", RUN_HEAVY, "-> the 15 GB CSV is NOT touched in this notebook.")


RUN_HEAVY = False -> the 15 GB CSV is NOT touched in this notebook.


In [3]:
# --- Load the two compact files (instant; these are what the notebook uses) ---
t0 = time.time()
stats = pl.read_parquet(STATS)   # (player, games) for ALL players, full graph
edges = pl.read_parquet(EDGES)   # (src_id, dst_id, w) for the top-5000 subgraph

print(f"loaded in {time.time()-t0:.2f}s")
print(f"player_stats : {stats.shape[0]:,} players      columns={stats.columns}")
print(f"edges_top    : {edges.shape[0]:,} directed edges columns={edges.columns}")

# Sanity check: summing every player's games equals the full 429M edge count,
# because each game contributes one row per direction = one to a src tally.
print("sum of all players' games (should equal ~429.7M):",
      f"{stats['games'].sum():,}")
stats.head(5)


loaded in 0.08s
player_stats : 519,583 players      columns=['player', 'games']
edges_top    : 7,989,314 directed edges columns=['src_id', 'dst_id', 'w']
sum of all players' games (should equal ~429.7M): 429,747,476


player,games
str,u32
"""mscp""",1622052
"""inemuri""",1410447
"""IFDThor""",856922
"""GriffyJr""",798206
"""GriffySr""",738643


## 2. Top-10 most central players (two notions, compared)

"**Centrality**" is a family of scores that try to capture *how important a node
is* inside a network. There is no single correct definition — different notions
of "important" give different rankings — so the task asks for **at least two**
and a comparison. I use three, of two fundamentally different kinds.

### 2a. PRIMARY measure — weighted degree = total games (EXACT, full graph)

The **degree** of a node is normally "how many edges touch it". When edges carry
weights, the **weighted degree** (also called **strength**) is the *sum* of those
weights. In this network an edge's weight is a count of games, so a player's
weighted degree is simply their **total number of games** — which is exactly the
`games` column we computed over the entire 429-million-edge graph.

> **Tiny example.** Suppose Alice played Bob 3 times and Carol 5 times. Alice's
> *plain* degree is 2 (two distinct opponents). Alice's *weighted* degree
> (strength) is 3 + 5 = 8 (total games). Here we report the weighted version.

This is our most trustworthy ranking: it is exact and uses every game ever
played. The intuition is "the most central players are the ones who show up the
most", which for an activity network is a very reasonable first definition of
importance.


In [4]:
# --- PRIMARY: top-10 by weighted degree (total games), EXACT on full graph ---
top10_games = stats.head(10)          # already sorted descending by 'games'
print("TOP-10 most central by WEIGHTED DEGREE = total games (FULL 429M graph):\n")
for rank, (p, g) in enumerate(zip(top10_games["player"].to_list(),
                                  top10_games["games"].to_list()), 1):
    print(f"{rank:>2}. {p:<14} {g:>10,} games")


TOP-10 most central by WEIGHTED DEGREE = total games (FULL 429M graph):

 1. mscp            1,622,052 games
 2. inemuri         1,410,447 games
 3. IFDThor           856,922 games
 4. GriffyJr          798,206 games
 5. GriffySr          738,643 games
 6. callipygian       505,557 games
 7. BabyLurking       390,377 games
 8. parrot            314,696 games
 9. AndreD            283,455 games
10. Uirapuru          272,717 games


### 2b. STRUCTURAL measures — PageRank, degree, strength on the top-5000 subgraph

The game-count ranking only knows *how much* someone played, not *whom against*.
**Structural** centrality looks at the *shape of the connections*. We build the
top-5000 graph in `igraph` (fast C library) and compute three things.

**Choice: we treat the graph as UNDIRECTED.** The raw data is directed (it stores
`(a,b)` and `(b,a)` separately), but a chess game between two people has no real
"direction" — A vs B is the same encounter as B vs A. So we **collapse** each
pair of opposite-direction edges into one undirected edge whose weight is the sum
(the total games that pair played against each other). This halves the edge count
(~8.0M directed -> ~4.0M undirected) and matches the real-world meaning. We
document this so it is a deliberate, stated choice.

The three structural scores:

- **PageRank** — originally Google's web-page ranking idea. Picture a random
  player who keeps hopping from opponent to opponent: from where they are now,
  they jump to one of that player's opponents, more likely along the *heavier*
  (more-games) edges. PageRank is the long-run fraction of time spent at each
  player. You score high if **many** players (especially other high-scoring
  players) play you a lot. It rewards being well-connected to other well-connected
  hubs, not just being busy.
  > *Tiny example:* a player who has only one opponent, but that opponent is the
  > single busiest hub in the network, can out-rank a player with several minor
  > opponents — because PageRank "flows" importance in from important neighbours.

- **Degree (here = number of distinct opponents)** — how many *different* people
  this player faced *within the top-5000 set*. This rewards *variety* of
  opponents, regardless of how many games each.

- **Strength (weighted degree within the subgraph)** — total games played
  *against other top-5000 players*. Like the primary measure, but restricted to
  the busy core, so it can differ from the full-graph games count.

**Performance note (important).** The subgraph has millions of edges. PageRank,
degree, and strength are cheap on `igraph` (well under a second). But **path-based
measures like betweenness and closeness are O(V·E)** — their cost grows with the
number of nodes *times* the number of edges — so on ~4 million edges they would
run for a very long time. We therefore **do not** compute betweenness/closeness on
the full subgraph (the task explicitly warns against it). If a path measure were
needed, the right move is to first shrink the graph (keep only heavy edges, or the
top ~300–500 players) and say so. Here PageRank already gives us a strong
path-flavoured ranking cheaply.


In [5]:
# --- Collapse the directed top-5000 edges into one UNDIRECTED weighted graph ---
# For each unweighted pair we sort the two usernames into (a, b) with a <= b so
# that (X,Y) and (Y,X) map to the SAME key, then sum their game counts.
t1 = time.time()
pe = (edges
      .with_columns([
          pl.min_horizontal("src_id", "dst_id").alias("a"),
          pl.max_horizontal("src_id", "dst_id").alias("b"),
      ])
      .group_by(["a", "b"]).agg(pl.col("w").sum().alias("w"))
      .filter(pl.col("a") != pl.col("b")))    # drop any self-games
print(f"undirected edges: {pe.shape[0]:,} "
      f"(down from {edges.shape[0]:,} directed)  [{time.time()-t1:.2f}s]")

# Build the igraph graph. igraph wants integer vertex ids, so we map each
# username to an index, then attach the username back as a 'name' attribute.
verts = pl.concat([pe["a"], pe["b"]]).unique().to_list()
idx = {v: i for i, v in enumerate(verts)}
edge_pairs = [(idx[s], idx[d])
              for s, d in zip(pe["a"].to_list(), pe["b"].to_list())]

t2 = time.time()
g = igraph.Graph(n=len(verts), edges=edge_pairs, directed=False)
g.vs["name"]   = verts
g.es["weight"] = pe["w"].to_list()
print(f"igraph built: |V|={g.vcount():,}  |E|={g.ecount():,}  "
      f"[{time.time()-t2:.2f}s]")


undirected edges: 3,994,657 (down from 7,989,314 directed)  [1.42s]


igraph built: |V|=5,000  |E|=3,994,657  [2.41s]


In [6]:
# --- Compute the three structural centralities (all fast on igraph) ---
t3 = time.time()
pr       = g.pagerank(weights="weight")        # importance via random walk
deg      = g.degree()                          # distinct opponents in subgraph
strength = g.strength(weights="weight")        # total games vs top-5000 players
print(f"PageRank + degree + strength computed in {time.time()-t3:.2f}s")

# Map each top-5000 player back to their EXACT full-graph game count for context.
games_map = dict(zip(stats["player"].to_list(), stats["games"].to_list()))

df = pd.DataFrame({
    "player":     g.vs["name"],
    "pagerank":   pr,
    "degree":     deg,
    "strength":   strength,
})
df["games_full"] = df["player"].map(games_map)   # exact full-graph total games
print("structural results table shape:", df.shape)


PageRank + degree + strength computed in 0.77s


structural results table shape: (5000, 5)


In [7]:
# --- TOP-10 by PageRank (the main structural ranking) ---
pr_top = df.sort_values("pagerank", ascending=False).head(10).reset_index(drop=True)
pr_top.index = pr_top.index + 1
print("TOP-10 by PAGERANK (undirected top-5000 subgraph):\n")
print(pr_top[["player", "pagerank", "degree", "strength", "games_full"]]
      .to_string(float_format=lambda x: f"{x:,.5f}" if x < 1 else f"{x:,.0f}"))


TOP-10 by PAGERANK (undirected top-5000 subgraph):

         player  pagerank  degree  strength  games_full
1          mscp   0.00503    2683 1,161,538     1622052
2       inemuri   0.00396    2154   723,526     1410447
3       IFDThor   0.00325    1863   727,022      856922
4      GriffyJr   0.00275    2048   649,868      798206
5      GriffySr   0.00201    1825   462,440      738643
6        AndreD   0.00158    1624   411,984      283455
7      Uirapuru   0.00141    3210   334,558      272717
8    mrlighting   0.00138    2158   346,282      252289
9   BabyLurking   0.00129    1472   240,164      390377
10  callipygian   0.00122    1686   242,458      505557


In [8]:
# --- TOP-10 by DEGREE (most distinct opponents) and by STRENGTH ---
deg_top = df.sort_values("degree", ascending=False).head(10).reset_index(drop=True)
deg_top.index = deg_top.index + 1
print("TOP-10 by DEGREE = number of distinct opponents (within top-5000):\n")
print(deg_top[["player", "degree", "strength", "pagerank", "games_full"]]
      .to_string(float_format=lambda x: f"{x:,.5f}" if x < 1 else f"{x:,.0f}"))

print("\n" + "="*70 + "\n")

str_top = df.sort_values("strength", ascending=False).head(10).reset_index(drop=True)
str_top.index = str_top.index + 1
print("TOP-10 by STRENGTH = total games vs other top-5000 players:\n")
print(str_top[["player", "strength", "degree", "pagerank", "games_full"]]
      .to_string(float_format=lambda x: f"{x:,.5f}" if x < 1 else f"{x:,.0f}"))


TOP-10 by DEGREE = number of distinct opponents (within top-5000):

      player  degree  strength  pagerank  games_full
1    Heidrun    3690   116,688   0.00053      108494
2      blore    3526    74,918   0.00041       79688
3   andreasw    3517   107,504   0.00054      106793
4   felipedj    3459   163,782   0.00072      136267
5       Carl    3393    99,704   0.00049       93098
6     korrin    3383    33,092   0.00020       38970
7    monacan    3375   122,380   0.00066      128641
8     sphinx    3349    35,526   0.00021       41704
9    Pushkin    3345    78,670   0.00048      103966
10     naomi    3342   111,008   0.00062      117256




TOP-10 by STRENGTH = total games vs other top-5000 players:

        player  strength  degree  pagerank  games_full
1         mscp 1,161,538    2683   0.00503     1622052
2      IFDThor   727,022    1863   0.00325      856922
3      inemuri   723,526    2154   0.00396     1410447
4     GriffyJr   649,868    2048   0.00275      798206
5     GriffySr   462,440    1825   0.00201      738643
6       AndreD   411,984    1624   0.00158      283455
7   mrlighting   346,282    2158   0.00138      252289
8     Uirapuru   334,558    3210   0.00141      272717
9      theprof   282,488    1861   0.00115      193805
10     bakkrot   275,202    2725   0.00112      173674


### 2c. Comparing the rankings — do the notions agree?

Now the interesting part: **do different definitions of "central" pick the same
people?** We measure the *overlap* of the top-10 lists and the **Spearman rank
correlation** between the full scores.

(Spearman correlation = take two scores, replace each by its *rank* — 1st, 2nd,
3rd … — and see how well those ranks line up. It runs from +1 "identical ordering"
through 0 "no relation" to −1 "exactly reversed". We use ranks instead of raw
values because these scores live on wildly different scales.)


In [9]:
# --- Overlap of the top-10 lists, and Spearman rank correlations ---
set_games = set(stats.head(10)["player"].to_list())
set_pr    = set(df.sort_values("pagerank",  ascending=False).head(10)["player"])
set_deg   = set(df.sort_values("degree",    ascending=False).head(10)["player"])
set_str   = set(df.sort_values("strength",  ascending=False).head(10)["player"])

print("Overlap of TOP-10 lists (out of 10):")
print(f"  weighted-degree (games) vs PageRank : {len(set_games & set_pr)}")
print(f"  weighted-degree (games) vs strength : {len(set_games & set_str)}")
print(f"  weighted-degree (games) vs degree   : {len(set_games & set_deg)}")
print(f"  PageRank                vs degree   : {len(set_pr   & set_deg)}")

print("\nSpearman rank correlation across the 5000 subgraph players:")
sub = df.dropna()
for a, b in [("pagerank","games_full"), ("strength","games_full"),
             ("pagerank","strength"), ("degree","games_full"),
             ("degree","pagerank")]:
    rho = spearmanr(sub[a], sub[b]).correlation
    print(f"  {a:<10} vs {b:<11}: rho = {rho:+.3f}")


Overlap of TOP-10 lists (out of 10):
  weighted-degree (games) vs PageRank : 9
  weighted-degree (games) vs strength : 7
  weighted-degree (games) vs degree   : 0
  PageRank                vs degree   : 0

Spearman rank correlation across the 5000 subgraph players:
  pagerank   vs games_full : rho = +0.897
  strength   vs games_full : rho = +0.808
  pagerank   vs strength   : rho = +0.958
  degree     vs games_full : rho = +0.378
  degree     vs pagerank   : rho = +0.499


**What the comparison shows.**

- **PageRank and total-games agree almost perfectly at the top.** Their top-10
  lists share **9 of 10** names, and across all 5000 players their rank
  correlation is about **+0.90**. This is the empirical proof of the justification
  from Section 1: the structurally central players really are the most active
  ones, so restricting structural analysis to the busy core loses essentially
  nobody who would have topped the ranking. Strength agrees even more tightly with
  PageRank (rho ≈ **+0.96**), which makes sense — both reward heavy, well-connected
  play.

- **Plain degree (distinct opponents) tells a *different* story.** Its top-10 has
  **zero** overlap with the games top-10, and it correlates only weakly with the
  others (rho ≈ +0.4–0.5). The players with the *most distinct opponents* (e.g.
  `Heidrun`, `blore`, `andreasw`, each facing ~3,300–3,700 different people) are
  not the players with the most *games*. A natural reading: these are accounts that
  play a *huge variety* of opponents a *few* times each — exactly the fingerprint
  of automated pairing engines or popular "open challenge" accounts — whereas the
  games leaders (`mscp`, `inemuri`) rack up enormous totals by playing a somewhat
  narrower set of opponents *very* many times.
  > **Misconception to avoid:** "more games must mean more opponents." Not here.
  > Volume (strength) and variety (degree) are genuinely different axes of being
  > central, and this dataset separates them cleanly.

So the two notions we were asked to compare — exact **weighted degree** and
structural **PageRank** — broadly *agree* on who the kingpins are (good: it means
our answer is robust), while the extra **degree** measure usefully reveals a
*second* kind of central player (the high-variety accounts) that the volume
measures hide.


## 3. Visualising a meaningful part of the network

We obviously cannot draw 5,000 nodes and 4,000,000 edges — it would be an
unreadable black blob. The skill is choosing a **small, legible slice** that
still tells a true story. We take the **top-40 players by total games** and draw
only their **heaviest head-to-head rivalries**: we keep only player-pairs whose
*combined* game count is in the **top 20%** of pairs among those 40 players (an
"edge-weight threshold" — a cutoff that throws away the thin, faint edges so the
strong relationships stand out). Any player left with no surviving edge is
dropped, so the picture shows the connected heart of the elite.

**Visual encoding (how data maps to ink):**
- **Node size and colour** ∝ that player's **total games** over the full graph
  (bigger and brighter = more games). Colour uses the `plasma` colour map.
- **Edge width** ∝ the **number of games in that specific rivalry** (thicker =
  these two faced each other more often).
- **Layout:** a **force-directed ("spring") layout**. This is a physics
  simulation: edges act like springs pulling connected nodes together, and all
  nodes repel each other like magnets, so the algorithm settles into a tidy
  arrangement where tightly-linked players sit near each other. We fix the seed so
  the layout is reproducible.


In [10]:
# --- FIGURE 1: top-40 most active players + their heaviest rivalries ---
TOPN = 40
top_players = stats.head(TOPN)["player"].to_list()

# Restrict the undirected edges to pairs where BOTH ends are in the top-40.
slice_e = pe.filter(pl.col("a").is_in(top_players) & pl.col("b").is_in(top_players))

# Edge-weight threshold: keep only the heaviest 20% of those rivalries so the
# drawing stays readable.
THR = int(np.quantile(slice_e["w"].to_numpy(), 0.80))
slice_e = slice_e.filter(pl.col("w") >= THR)
print(f"top-{TOPN}: {slice_e.shape[0]} rivalries kept (combined games >= {THR:,})")

# Build a small networkx graph just for drawing.
G = nx.Graph()
for p in top_players:
    G.add_node(p, games=games_map[p])
for a, b, w in zip(slice_e["a"].to_list(), slice_e["b"].to_list(),
                   slice_e["w"].to_list()):
    G.add_edge(a, b, weight=w)
G.remove_nodes_from(list(nx.isolates(G)))      # drop players with no kept edge
print(f"drawing {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

pos = nx.spring_layout(G, k=1.5, iterations=300, seed=42, weight=None)
ng  = np.array([G.nodes[n]["games"] for n in G.nodes()])
nsz = 150 + 1800 * (ng - ng.min()) / (ng.max() - ng.min() + 1e-9)
ew  = np.array([G[u][v]["weight"] for u, v in G.edges()])
ewd = 0.3 + 5.0 * (ew - ew.min()) / (ew.max() - ew.min() + 1e-9)

fig, ax = plt.subplots(figsize=(15, 12))
nx.draw_networkx_edges(G, pos, width=ewd, alpha=0.30, edge_color="#3366aa", ax=ax)
nodes = nx.draw_networkx_nodes(G, pos, node_size=nsz, node_color=ng, cmap="plasma",
                               alpha=0.9, linewidths=0.5, edgecolors="white", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold", ax=ax)
cbar = fig.colorbar(nodes, ax=ax, shrink=0.55, pad=0.01)
cbar.set_label("Total games over full 429M-edge graph")
ax.set_title(
    f"FICS chess: the {G.number_of_nodes()} most-active players and their "
    f"heaviest head-to-head rivalries\n(only player-pairs with >= {THR:,} "
    f"combined games shown; node size & colour proportional to total games, "
    f"edge width proportional to games in that rivalry)", fontsize=12)
ax.axis("off")
path1 = na.save_fig(fig, "partA6_chess_top_players.png")
plt.close(fig)
print("saved:", path1)


top-40: 65 rivalries kept (combined games >= 920)
drawing 30 nodes and 65 edges


saved: /home/mickaelz/Network analysis/figures/partA6_chess_top_players.png


![Top players and rivalries](../../figures/partA6_chess_top_players.png)

**Reading this picture.** The biggest, brightest nodes — `mscp`, `inemuri`,
`IFDThor`, `GriffyJr`, `GriffySr` — are the game-count leaders and they sit at the
hubs of the busiest rivalries. The thickest single edge in the slice typically
links two superstars who have faced each other an enormous number of times (for
example the `GriffyJr`–`GriffySr` pairing, very plausibly two closely-linked
accounts that play each other constantly). Notice that several large nodes connect
out to smaller satellites: these are top players whose heaviest opponent is *not*
themselves a top-40 player. The thresholding is what makes the structure legible —
without it, the 40 elite players would be joined by hundreds of faint edges into
an unreadable mesh.


In [11]:
# --- FIGURE 2: the games (weighted-degree) distribution on log-log axes ---
# Real social/interaction networks are 'heavy-tailed': a few nodes have enormous
# degree, most have tiny degree. On log-log axes that shows up as a near-straight,
# slowly-decaying line. We show it two ways: a log-binned histogram and a CCDF.
games = stats["games"].to_numpy()
games = games[games > 0]

# (left) log-binned histogram: bucket edges spaced evenly on a log scale.
bins = np.logspace(np.log10(games.min()), np.log10(games.max()), 40)
hist, be = np.histogram(games, bins=bins)
centers = np.sqrt(be[:-1] * be[1:])
m = hist > 0

# (right) complementary CDF: P(X >= x) = fraction of players with at least x games.
xs = np.sort(games)
ccdf = 1.0 - np.arange(len(xs)) / len(xs)

fig2, axs = plt.subplots(1, 2, figsize=(15, 6))
axs[0].loglog(centers[m], hist[m], "o-", color="#cc3333", ms=5)
axs[0].set_xlabel("Total games per player (weighted degree) — log scale")
axs[0].set_ylabel("Number of players in bin — log scale")
axs[0].set_title("Log-binned games distribution")
axs[1].loglog(xs, ccdf, color="#3366cc")
axs[1].set_xlabel("Total games per player x — log scale")
axs[1].set_ylabel("Fraction of players with >= x games")
axs[1].set_title("Complementary CDF (heavy-tail view)")
fig2.suptitle(
    f"FICS chess full graph: distribution of total-games-per-player across all "
    f"{stats.height:,} players\nA near-straight line on log-log axes is the "
    f"signature of a heavy-tailed (scale-free-like) network", fontsize=12)
path2 = na.save_fig(fig2, "partA6_chess_degree_distribution.png")
plt.close(fig2)
print("saved:", path2)

print(f"\nmedian games per player : {int(np.median(games)):,}")
print(f"mean   games per player : {games.mean():,.1f}")
print(f"max    games (one player): {int(games.max()):,}")
top1pct = np.sort(games)[::-1][:int(0.01*len(games))].sum()
print(f"share of ALL games held by the busiest 1% of players: "
      f"{100*top1pct/games.sum():.1f}%")


saved: /home/mickaelz/Network analysis/figures/partA6_chess_degree_distribution.png

median games per player : 13
mean   games per player : 827.1
max    games (one player): 1,622,052
share of ALL games held by the busiest 1% of players: 43.0%


![Games distribution](../../figures/partA6_chess_degree_distribution.png)

**Reading this picture.** Both panels say the same thing in two ways. The
distribution is **extremely heavy-tailed**: the *median* player has only about a
dozen games, yet the busiest single player has **over 1.6 million**, and the
busiest **1% of players account for roughly 43% of all games ever played**. The
near-straight downward line on log-log axes is the classic look of a
**scale-free-like** network — one with no "typical" scale, where a tiny elite of
super-active hubs coexists with a vast crowd of casual players. This is exactly
why a *top-K* induced subgraph is the right tool: the network's structure is
dominated by that small hub elite, so capturing the busiest few thousand players
captures the part that drives the centrality results.


## 4. How I solved this task

**What I did.** I found the top-10 most central FICS chess players under three
centrality notions and drew a readable slice of the busy core.

**The methods and why.**
1. **Streaming aggregation (polars).** The 429-million-edge / 15.4 GB graph cannot
   be loaded whole, so it was pre-summarised by reading the CSV once in
   memory-bounded chunks (`scan_csv(...).collect(engine="streaming")`) as a SLURM
   batch job. Grouping the doubled edge list by the source column gives **exact**
   per-player game counts (weighted degree) over the *entire* graph; a second
   streaming pass extracts the **induced subgraph on the top-5000 busiest
   players** for structural work. I chose streaming because it is the only way to
   get *exact* full-graph statistics on a machine that cannot hold the graph.
2. **Centrality (igraph + the exact counts).** The **primary** ranking is
   weighted degree = total games, exact on the full graph. The **structural**
   rankings — PageRank, degree (distinct opponents), and strength — run on the
   undirected top-5000 subgraph in igraph (C-fast, sub-second). I treated the
   graph as undirected because a chess game has no direction, and I deliberately
   avoided betweenness/closeness because their O(V·E) cost would hang on millions
   of edges.
3. **Visualisation (networkx + matplotlib).** I drew the top-40 players keeping
   only their heaviest 20% of rivalries (edge-weight threshold), with node
   size/colour = total games and edge width = rivalry games, on a reproducible
   force-directed layout. A second log-log plot shows the heavy-tailed games
   distribution of the full graph.

**What the main result means.** The kingpins of FICS are `mscp` (~1.62M games) and
`inemuri` (~1.41M games), followed by `IFDThor`, `GriffyJr`, and `GriffySr`. The
exact volume ranking and the structural PageRank ranking **agree on 9 of their top
10** (rank correlation ≈ +0.90), which both validates the answer and proves that
analysing only the busy core was sound. Plain degree reveals a *different* elite —
high-variety accounts that face thousands of distinct opponents a few times each —
showing that "central" has more than one meaning here.

**Limitations and assumptions.**
- The structural measures (PageRank, degree, strength) are computed on the
  **top-5000 induced subgraph**, not the full graph. We justified this (the
  central players are the active ones, confirmed by the 9/10 overlap), but a player
  whose importance came entirely from connecting *low-activity* players would be
  invisible to it. The primary game-count ranking has no such limitation — it is
  exact and global.
- We **collapsed direction**; any genuinely directional effect in the raw data is
  intentionally discarded as not meaningful for chess encounters.
- The data has no game **outcomes** or **ratings**, so "central" here means
  "central by activity/structure", not "strongest player". The handful of accounts
  that dominate the counts are very likely **bots or engine/pairing accounts**
  rather than human grandmasters.
- We did **not** compute path-based centralities (betweenness/closeness) on the
  full subgraph for the performance reasons stated; doing so would require first
  reducing to a few hundred heavy-edge nodes.

**Data source:** <http://dynamics.cs.washington.edu/nobackup/chess/fcis.tar.gz>
**Libraries:** polars, python-igraph, networkx, numpy, pandas, scipy, matplotlib.
